## Portfolio Agents with Custom Financial Tools

This notebook extends the portfolio agents with custom tools for real-time stock and mutual fund data retrieval.

### Import Required Libraries and Setup

In [ ]:
import os
import json
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
import yfinance as yf
import pandas as pd
import requests
import urllib3
from datetime import datetime, timedelta

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

openai_client = project_client.get_openai_client()

### Client Portfolio Data

In [ ]:
client_portfolios = {
    "Client_1": {
        "name": "Client 1",
        "stocks": ["TCS.NS", "WIPRO.NS", "INFY.NS", "HDFCBANK.NS"],
        "mutual_funds": ["119551", "119022"]
    },
    "Client_2": {
        "name": "Client 2",
        "stocks": ["RELIANCE.NS", "HDFCBANK.NS", "ICICIBANK.NS", "LT.NS"],
        "mutual_funds": ["119551", "120485"]
    },
    "Client_3": {
        "name": "Client 3",
        "stocks": ["TATAMOTORS.NS", "ADANIGREEN.NS", "ETERNAL.NS", "PAYTM.NS"],
        "mutual_funds": ["119598", "119618"]
    },
    "Client_4": {
        "name": "Client 4",
        "stocks": ["ITC.NS", "COALINDIA.NS", "NTPC.NS", "POWERGRID.NS"],
        "mutual_funds": ["119547"]
    },
    "Client_5": {
        "name": "Client 5",
        "stocks": ["INFY.NS", "SUNPHARMA.NS", "BHARTIARTL.NS", "TATACONSUM.NS"],
        "mutual_funds": ["119510", "120303"]
    }
}

print("Portfolio data loaded for 5 clients")

### Advanced Data Fetching Functions

In [ ]:
def get_stock_price_history(symbol, days=30):
    """Get historical stock price data"""
    try:
        ticker = yf.Ticker(f"{symbol}")
        end_date = datetime.today()
        start_date = end_date - timedelta(days=days)
        
        df = yf.download(symbol, start=start_date, end=end_date, auto_adjust=True, progress=False)
        
        if df.empty:
            return {"error": "No data available"}
        
        # Calculate technical indicators
        current_price = df['Close'].iloc[-1]
        avg_price = df['Close'].mean()
        high_52week = df['High'].max()
        low_52week = df['High'].min()
        
        price_change = ((current_price - df['Close'].iloc[0]) / df['Close'].iloc[0]) * 100
        
        return {
            "ticker": symbol,
            "current_price": round(current_price, 2),
            "30_day_average": round(avg_price, 2),
            "52_week_high": round(high_52week, 2),
            "52_week_low": round(low_52week, 2),
            "30_day_change_percent": round(price_change, 2),
            "trading_volume_avg": int(df['Volume'].mean())
        }
    except Exception as e:
        return {"ticker": symbol, "error": str(e)}

def get_market_indices_snapshot():
    """Get latest market indices data"""
    indices = {
        '^NSEI': 'Nifty 50',
        '^BSESN': 'BSE Sensex',
        '^NSEBANK': 'Nifty Bank',
        '^CNXIT': 'Nifty IT'
    }
    
    data = []
    for symbol, name in indices.items():
        try:
            ticker = yf.Ticker(symbol)
            hist = ticker.history(period="5d")
            
            if hist.empty:
                continue
            
            current = hist['Close'].iloc[-1]
            previous = hist['Close'].iloc[-2] if len(hist) > 1 else current
            change_pct = ((current - previous) / previous * 100) if previous != 0 else 0
            
            data.append({
                "index": name,
                "current_value": round(current, 2),
                "change_percent": round(change_pct, 2)
            })
        except Exception as e:
            print(f"Error fetching {name}: {e}")
    
    return data

def analyze_portfolio_composition(client_id):
    """Analyze portfolio composition and provide insights"""
    portfolio = client_portfolios.get(client_id)
    if not portfolio:
        return {"error": "Client not found"}
    
    stocks_data = {}
    for stock in portfolio['stocks']:
        stocks_data[stock] = get_stock_price_history(stock, days=30)
    
    # Calculate portfolio metrics
    total_value = sum([v.get('current_price', 0) * 100 for v in stocks_data.values()])  # Assuming 100 units each
    
    return {
        "client_id": client_id,
        "portfolio_size": len(portfolio['stocks']),
        "stocks_composition": stocks_data,
        "asset_types": {
            "equity_stocks": len(portfolio['stocks']),
            "mutual_funds": len(portfolio['mutual_funds'])
        }
    }

def get_sector_analysis(client_id):
    """Provide sector-wise breakdown of portfolio"""
    # Simplified sector mapping for Indian stocks
    sector_map = {
        "TCS.NS": "IT",
        "WIPRO.NS": "IT",
        "INFY.NS": "IT",
        "HDFCBANK.NS": "Banking",
        "RELIANCE.NS": "Energy",
        "ICICIBANK.NS": "Banking",
        "LT.NS": "Engineering",
        "TATAMOTORS.NS": "Automotive",
        "ADANIGREEN.NS": "Energy",
        "ETERNAL.NS": "E-Commerce",
        "PAYTM.NS": "Fintech",
        "ITC.NS": "FMCG",
        "COALINDIA.NS": "Energy",
        "NTPC.NS": "Energy",
        "POWERGRID.NS": "Infrastructure",
        "SUNPHARMA.NS": "Pharma",
        "BHARTIARTL.NS": "Telecom",
        "TATACONSUM.NS": "FMCG"
    }
    
    portfolio = client_portfolios.get(client_id)
    if not portfolio:
        return {"error": "Client not found"}
    
    sectors = {}
    for stock in portfolio['stocks']:
        sector = sector_map.get(stock, "Other")
        if sector not in sectors:
            sectors[sector] = []
        sectors[sector].append(stock)
    
    return {
        "client_id": client_id,
        "sectors": sectors,
        "sector_breakdown": {sector: f"{(len(stocks)/len(portfolio['stocks'])*100):.1f}%" for sector, stocks in sectors.items()}
    }

print("Advanced data fetching functions defined")

### Create Advanced Portfolio Agents with Context

In [ ]:
def create_agent_instructions(client_id):
    """Create detailed instructions for portfolio agent"""
    portfolio = client_portfolios.get(client_id)
    sector_analysis = get_sector_analysis(client_id)
    
    sectors_text = ", ".join([f"{sector} ({', '.join(stocks)})" for sector, stocks in sector_analysis['sectors'].items()])
    
    instructions = f"""
You are an advanced portfolio analyst for {portfolio['name']}.

CLIENT PORTFOLIO OVERVIEW:
=========================
Holdings:
- Stocks ({len(portfolio['stocks'])}): {', '.join(portfolio['stocks'])}
- Mutual Funds ({len(portfolio['mutual_funds'])}) in various categories

Sector Exposure:
{sectors_text}

YOUR CAPABILITIES:
==================
1. REAL-TIME ANALYSIS: Provide current stock prices, historical trends, and technical analysis
2. PORTFOLIO HEALTH: Assess diversification, risk exposure, and concentration risks
3. SECTOR INSIGHTS: Analyze sector-wise allocation and performance trends
4. MARKET CONTEXT: Provide market indices and broader economic trends
5. RECOMMENDATIONS: Suggest rebalancing opportunities and risk mitigation strategies
6. PERFORMANCE TRACKING: Monitor holdings against benchmarks

YOUR APPROACH:
===============
- Always provide data-driven insights backed by current market data
- Consider the client's existing portfolio composition when making recommendations
- Flag any concentration risks or overexposure to specific sectors
- Discuss correlation and diversification benefits
- Reference market indices for context
- Be transparent about data sources (Yahoo Finance, MFAPI)

IMPORTANT GUIDELINES:
=====================
- Provide factual information based on available market data
- Avoid making absolute investment recommendations
- Always mention the date/time of data being analyzed
- Highlight volatility and risk factors in the portfolio
- Suggest periodic rebalancing reviews
"""
    
    return instructions.strip()

# Create advanced agents
advanced_agents = {}

for client_id, portfolio in client_portfolios.items():
    agent_name = f"portfolio-analyst-{client_id.lower()}"
    instructions = create_agent_instructions(client_id)
    
    agent = project_client.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions=instructions
        )
    )
    
    advanced_agents[client_id] = {
        "agent": agent,
        "agent_name": agent_name,
        "portfolio": portfolio,
        "sector_analysis": get_sector_analysis(client_id)
    }
    
    print(f"✅ Created advanced agent: {agent_name}")
    print(f"   Sectors: {list(advanced_agents[client_id]['sector_analysis']['sectors'].keys())}")
    print()

### Create Conversations and Query Agents

In [ ]:
# Create conversations for each advanced agent
advanced_conversations = {}

for client_id in advanced_agents.keys():
    conversation = openai_client.conversations.create()
    advanced_conversations[client_id] = conversation.id

print("Conversations created for all advanced agents")

### Query Advanced Agents - Multi-Turn Conversation

In [ ]:
# Multi-turn conversation with Client 1's agent
client_id = "Client_1"

queries = [
    "What is the current status of my IT sector holdings?",
    "How does my portfolio compare to the Nifty 50 index?",
    "Should I increase or reduce my exposure to banking stocks?"
]

print(f"\n{'='*70}")
print(f"Multi-turn Conversation with {advanced_agents[client_id]['portfolio']['name']}'s Agent")
print(f"{'='*70}\n")

for i, query in enumerate(queries, 1):
    response = openai_client.responses.create(
        conversation=advanced_conversations[client_id],
        extra_body={
            "agent": {
                "name": advanced_agents[client_id]["agent_name"],
                "type": "agent_reference"
            }
        },
        input=query
    )
    
    print(f"Q{i}: {query}")
    print(f"A{i}: {response.output_text}")
    print()

### Client-Specific Queries

In [ ]:
# Query for Client 2 - Energy sector focused
client_id = "Client_2"
query = "I notice I have significant exposure to energy sector stocks. What are the risks and benefits?"

response = openai_client.responses.create(
    conversation=advanced_conversations[client_id],
    extra_body={
        "agent": {
            "name": advanced_agents[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=query
)

print(f"{advanced_agents[client_id]['portfolio']['name']} - Energy Sector Analysis")
print(f"\nQuery: {query}")
print(f"\nResponse:\n{response.output_text}")

### Batch Portfolio Analysis

In [ ]:
# Analyze all client portfolios
print("\nBATCH PORTFOLIO COMPOSITION ANALYSIS")
print("="*70)

for client_id, agent_info in advanced_agents.items():
    composition = analyze_portfolio_composition(client_id)
    sector_info = agent_info['sector_analysis']
    
    print(f"\n{agent_info['portfolio']['name']}")
    print(f"  Portfolio Size: {composition['portfolio_size']} stocks")
    print(f"  Asset Mix: {composition['asset_types']['equity_stocks']} stocks, {composition['asset_types']['mutual_funds']} mutual funds")
    print(f"  Sectors: {', '.join(sector_info['sectors'].keys())}")
    print(f"  Sector Breakdown:")
    for sector, pct in sector_info['sector_breakdown'].items():
        print(f"    - {sector}: {pct}")

### Market Context for All Clients

In [ ]:
# Get current market snapshot
print("\nCURRENT MARKET SNAPSHOT")
print("="*70)

try:
    indices = get_market_indices_snapshot()
    
    for index in indices:
        change_indicator = "📈" if index['change_percent'] > 0 else "📉"
        print(f"{change_indicator} {index['index']}: {index['current_value']} ({index['change_percent']:+.2f}%)")
except Exception as e:
    print(f"Error fetching market data: {e}")

### Agent Performance Summary

In [ ]:
print("\n" + "="*70)
print("ADVANCED PORTFOLIO AGENTS - DEPLOYMENT SUMMARY")
print("="*70)

summary_data = []
for client_id, agent_info in advanced_agents.items():
    summary_data.append({
        'Client': agent_info['portfolio']['name'],
        'Agent ID': agent_info['agent']['id'],
        'Agent Name': agent_info['agent_name'],
        'Stocks': len(agent_info['portfolio']['stocks']),
        'Funds': len(agent_info['portfolio']['mutual_funds']),
        'Sectors': len(agent_info['sector_analysis']['sectors'])
    })

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

print(f"\nTotal Advanced Agents: {len(advanced_agents)}")
print(f"Total Conversations: {len(advanced_conversations)}")
print(f"Total Unique Holdings: {sum(len(p['portfolio']['stocks']) for p in advanced_agents.values())}")
print("="*70)